# Кластеризация ЕГЭ (4 если сделаны все задачи)

Рядом лежат данные с координатами точек. Везде используется Евклидово расстояние. Кластером считается набор не менее чем из 30 точек связанных друг с другом. Аномалия это точка находящаяся на расстоянии более 1 от любого кластера.

* Постройте Распределение точек
* Напишите руками DBSCAN и обработайте им все файлы
* Файл 0.xls также решите руками
* Постройте Распределение точек, отметьте принадлежность кластеров цветами
* Отметьте Аномалии отдельным цветом
* Найдите среди в каждом кластере точку расстояние от которой до всех остальных минимально
* Выведите два числа - Среднее абсцисс и ординат центроидов кластеров * 10000

# Кластеризация (1 за каждый алгоритм на всех данных)

На предложенных распределениях данных проверьте предложенные алгоритмы. Постройте графики кластеризации для каждой пары алгоритм-данные, разные кластеры покрасьте разным цветом. Воспользуйтесь sklearn реализациями. Параметры кластеризации для разных алгоритмов подберите такие, чтобы алгоритмы можно было сравнивать (по возможности одинаковое количество кластеров и т.д.)

In [ ]:
# Импорты
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
import os

# Ручная реализация DBSCAN
def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b) ** 2))

def dbscan_manual(points, eps=1.0, min_pts=30):
    """
    points: numpy array of shape (n, 2)
    eps: радиус соседства
    min_pts: минимальное число точек для образования кластера
    Возвращает: массив меток (-1 - шум)
    """
    n = len(points)
    labels = np.full(n, -1)
    cluster_id = 0

    def region_query(idx):
        """Найти все точки в пределах eps от точки idx"""
        neighbors = []
        for j in range(n):
            if euclidean_distance(points[idx], points[j]) <= eps:
                neighbors.append(j)
        return neighbors

    def expand_cluster(idx, neighbors):
        """Расширить кластер"""
        labels[idx] = cluster_id
        seeds = list(neighbors)
        k = 0
        while k < len(seeds):
            current = seeds[k]
            if labels[current] == -1:
                labels[current] = cluster_id
                current_neighbors = region_query(current)
                if len(current_neighbors) >= min_pts:
                    for nb in current_neighbors:
                        if labels[nb] == -1:
                            seeds.append(nb)
            k += 1

    for i in range(n):
        if labels[i] != -1:
            continue
        neighbors_i = region_query(i)
        if len(neighbors_i) < min_pts:
            continue  # оставляем -1
        cluster_id += 1
        expand_cluster(i, neighbors_i)

    return labels

# Чтение всех Excel-файлов
file_list = glob.glob("*.xls") + glob.glob("*.xlsx")
print(f"Найдено файлов: {len(file_list)}")

all_centroids = []  # для сбора всех центроидов

# Обработка каждого файла
for file in sorted(file_list):
    print(f"\nОбработка файла: {file}")
    # Загрузка данных (предполагаем, что координаты в колонках 'x' и 'y')
    df = pd.read_excel(file)
    points = df[['x', 'y']].values

    # Применяем DBSCAN
    labels = dbscan_manual(points, eps=1.0, min_pts=30)

    # Подсчёт статистики
    unique_labels = set(labels)
    n_clusters = len(unique_labels - {-1})
    n_noise = np.sum(labels == -1)
    print(f"Кластеров: {n_clusters}, шумовых точек: {n_noise}")

    # Визуализация
    plt.figure(figsize=(8, 6))
    scatter = plt.scatter(points[:, 0], points[:, 1], c=labels, cmap='viridis',
                          s=30, edgecolors='k', linewidth=0.5)
    plt.colorbar(scatter, label='Cluster ID')
    plt.title(f"DBSCAN на {file} (eps=1.0, min_pts=30)")
    plt.xlabel('x')
    plt.ylabel('y')
    plt.grid(True)
    plt.show()

    # Нахождение центральной точки для каждого кластера
    centroids = []
    for lbl in unique_labels:
        if lbl == -1:
            continue
        cluster_points = points[labels == lbl]
        # Вычисляем точку с минимальной суммой расстояний до остальных в кластере
        best_idx = None
        best_sum = float('inf')
        for i, p in enumerate(cluster_points):
            dist_sum = sum(euclidean_distance(p, q) for q in cluster_points if not np.array_equal(p, q))
            if dist_sum < best_sum:
                best_sum = dist_sum
                best_idx = i
        central_point = cluster_points[best_idx]
        centroids.append(central_point)
        all_centroids.append(central_point)
        print(f"Кластер {lbl} – центральная точка: {central_point}")

    # Если нужно, можно также отметить аномалии отдельно (на графике они уже другим цветом)

# После обработки всех файлов вычисляем среднее по всем центроидам
if all_centroids:
    centroids_array = np.array(all_centroids)
    mean_centroid = np.mean(centroids_array, axis=0)
    result = mean_centroid * 10000
    print(f"\nСреднее по всем центроидам (x, y): {mean_centroid}")
    print(f"Результат * 10000: {result}")
else:
    print("Не найдено ни одного центроида.")